In [ ]:
# ---------------------------------------------------------------------------
# CELL 0: MODEL LOADER (Link from Input Dataset)
# ---------------------------------------------------------------------------
import os
import shutil

# 1. Configuration
# Check your "Input" tab. The folder usually matches your dataset URL slug.
# Example: spw800621/sage-tachibana-svc -> sage-tachibana-svc
INPUT_DATASET_DIR = "/kaggle/input/sage-tachibana-svc" 
WORKING_MODELS_DIR = "/kaggle/working/svc_models"

os.makedirs(WORKING_MODELS_DIR, exist_ok=True)

print(f"🔗 Linking models from {INPUT_DATASET_DIR}...")

if os.path.exists(INPUT_DATASET_DIR):
    linked_count = 0
    freed_space = 0
    
    # Iterate through every model in the Read-Only Input
    for folder_name in os.listdir(INPUT_DATASET_DIR):
        src_path = os.path.join(INPUT_DATASET_DIR, folder_name)
        dst_path = os.path.join(WORKING_MODELS_DIR, folder_name)
        
        # We only care about directories (model folders)
        if os.path.isdir(src_path):
            
            # CLEANUP: If a physical folder exists in Working, delete it to save space
            if os.path.exists(dst_path):
                if os.path.islink(dst_path):
                    # It's already a link, ignore
                    pass
                else:
                    # It's a real folder occupying space! Delete it!
                    print(f"   🧹 Removing physical copy of '{folder_name}' to free space...")
                    shutil.rmtree(dst_path)
                    freed_space += 1

            # LINK: Create the symlink if it doesn't exist
            if not os.path.exists(dst_path):
                os.symlink(src_path, dst_path)
                linked_count += 1
                # print(f"   🔗 Linked: {folder_name}")
    
    print(f"✅ Success.")
    print(f"   - Replaced {freed_space} physical folders.")
    print(f"   - Active Links: {linked_count}")
    print(f"   - Disk Usage for these models: 0 MB")

else:
    print(f"❌ Error: Dataset not found at {INPUT_DATASET_DIR}")
    print("   Please check the exact folder name in the /kaggle/input/ directory.")

In [ ]:
# ---------------------------------------------------------------------------
# CHAMELEON MODEL LOADER (Link from Input Dataset)
# ---------------------------------------------------------------------------
import os
import shutil
import sys

# -----------------------
# 1) Configuration
# -----------------------
# Input dataset folder (usually matches the dataset slug under /kaggle/input/)
INPUT_SLUG = "sage-chameleon-svc"                       # adjust if the dataset folder name differs
INPUT_DATASET_DIR = os.path.join("/kaggle/input", INPUT_SLUG)

# Where we want symlinks to live (working area)
WORKING_MODELS_DIR = "/kaggle/working/svc_models"

# The exact model folder names you want linked (Chameleon list)
TARGET_FOLDERS = [
    "agent-47", "edi", "ezio-auditore", "fallenshadow", "fujikura-uruka",
    "hal-9000", "martin-sheen", "pipkin-pippa", "robert-osborne",
    "tali", "tenma-maemi", "vegeta"
]

# If True: don't actually change filesystem, just print what would happen
DRY_RUN = False

# -----------------------
# 2) Safety & Setup
# -----------------------
if not os.path.exists(INPUT_DATASET_DIR):
    print(f"❌ Error: Input dataset directory not found: {INPUT_DATASET_DIR}")
    print("   Check the dataset slug under /kaggle/input/ and adjust INPUT_SLUG if necessary.")
    sys.exit(1)

os.makedirs(WORKING_MODELS_DIR, exist_ok=True)
print(f"🔗 Linking models from {INPUT_DATASET_DIR} -> {WORKING_MODELS_DIR}")
print(f"   - Targets: {len(TARGET_FOLDERS)} models")
print(f"   - DRY_RUN = {DRY_RUN}")
print()

# -----------------------
# 3) Operations
# -----------------------
linked_count = 0
replaced_physical = 0
replaced_symlink = 0
skipped_not_found = 0
errors = []

for name in TARGET_FOLDERS:
    src = os.path.join(INPUT_DATASET_DIR, name)
    dst = os.path.join(WORKING_MODELS_DIR, name)

    # Only consider real directories at the input (os.path.isdir follows symlinks)
    if not os.path.isdir(src):
        print(f"   ⚠️  Skip (not a dir / missing): {name}")
        skipped_not_found += 1
        continue

    try:
        # If destination exists
        if os.path.exists(dst):
            # If it's an existing symlink
            if os.path.islink(dst):
                current_target = os.readlink(dst)
                # If symlink already points to requested src -> OK
                # Normalize absolute paths for fair comparison
                abs_current = os.path.abspath(os.path.join(os.path.dirname(dst), current_target))
                abs_src = os.path.abspath(src)
                if abs_current == abs_src:
                    print(f"   ✅ Already linked: {name}")
                else:
                    print(f"   🔁 Replacing incorrect symlink: {name}")
                    if not DRY_RUN:
                        os.remove(dst)
                        os.symlink(src, dst)
                    replaced_symlink += 1
                    linked_count += 1
            else:
                # It's a physical folder occupying space -> remove it and symlink
                print(f"   🧹 Removing physical folder and linking: {name}")
                if not DRY_RUN:
                    # use rmtree carefully
                    shutil.rmtree(dst)
                    os.symlink(src, dst)
                replaced_physical += 1
                linked_count += 1
        else:
            # dst does not exist -> create symlink
            print(f"   🔗 Linking: {name}")
            if not DRY_RUN:
                os.symlink(src, dst)
            linked_count += 1

    except Exception as e:
        errors.append((name, str(e)))
        print(f"   ❌ Error handling '{name}': {e}")

# -----------------------
# 4) Summary
# -----------------------
print("\n✅ Done.")
print(f"   - Linked / Re-created symlinks: {linked_count}")
print(f"   - Physical folders removed and replaced: {replaced_physical}")
print(f"   - Incorrect symlinks replaced: {replaced_symlink}")
print(f"   - Skipped (missing / not dirs): {skipped_not_found}")
if errors:
    print(f"   - Errors: {len(errors)} (see below)")
    for name, msg in errors:
        print(f"      • {name}: {msg}")

if DRY_RUN:
    print("\nNote: DRY_RUN=True, no filesystem changes were made.")

In [ ]:
# 1. Install System FFmpeg (Crucial for video encoding)
!apt-get update && apt-get install -y ffmpeg

# 2. Install Python bindings
#!pip install imageio[ffmpeg] imageio-ffmpeg

In [ ]:
# ---------------------------------------------------------------------------
# CELL 1: CONFIGURATION
# ---------------------------------------------------------------------------
import os
import sys

# --- 1. Hardcoded Dataset Path ---
# This matches the dataset slug you used in the deploy script: "sage-pyapi-code"
PYAPI_DATASET_NAME = "sage-pyapi-code"
PYAPI_SOURCE_DIR = f"/kaggle/input/{PYAPI_DATASET_NAME}"

# Quick validation to ensure it's mounted
if not os.path.exists(os.path.join(PYAPI_SOURCE_DIR, "main.py")):
    print(f"❌ Error: 'main.py' not found in {PYAPI_SOURCE_DIR}")
    print(f"   Did you name the dataset '{PYAPI_DATASET_NAME}'? Check the right sidebar.")
    sys.exit(1)

print(f"✅ PyAPI Source Configured: {PYAPI_SOURCE_DIR}")

# --- 2. Zrok Token Setup ---
ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
zrok_token = None

if os.path.isfile(ZROK_TOKEN_PATH):
    with open(ZROK_TOKEN_PATH, "r", encoding="utf-8", errors="ignore") as f:
        zrok_token = f.read().strip()

# Fallback to Secrets
if not zrok_token:
    from kaggle_secrets import UserSecretsClient
    try:
        zrok_token = UserSecretsClient().get_secret("zrok_token")
    except:
        pass

if not zrok_token:
    print("❌ Zrok Token not found! Ensure 'sage-zrok-token' dataset is added.")
    sys.exit(1)

print(f"✅ Zrok Token loaded.")

In [ ]:

# ---------------------------------------------------------------------------
# CELL 2: SYSTEM SETUP & LIBRARY INSTALLATION
# ---------------------------------------------------------------------------
import os
import shutil
import sys

# --- 1. Clean & Setup Workspace ---
if "PYTHONPATH" in os.environ: del os.environ["PYTHONPATH"]
!apt-get update && apt-get install -y libgl1-mesa-glx ffmpeg > /dev/null

WORK_DIR = "/kaggle/working/pyapi"
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
shutil.copytree(PYAPI_SOURCE_DIR, WORK_DIR)

# --- 2. Create Venv ---
VENV_PATH = os.path.join(WORK_DIR, "venv")
!python3 -m venv "{VENV_PATH}" --without-pip
VENV_BIN = os.path.join(VENV_PATH, "bin")
VENV_PIP = os.path.join(VENV_BIN, "pip")

# Bootstrap pip
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!{VENV_BIN}/python /tmp/get-pip.py > /dev/null

!{VENV_PIP} install wrapt --no-cache-dir

# --- 3. Install Requirements ---
print("📦 Installing Dependencies...")
!{VENV_PIP} install wheel setuptools "numpy<2" --no-cache-dir

# Install Server Requirements (This will install the WRONG torch initially)
req_path = os.path.join(WORK_DIR, "requirements_server_gpu.txt")
!{VENV_PIP} install -r "{req_path}" --no-cache-dir
!{VENV_PIP} install pyngrok nest_asyncio --no-cache-dir

print("✅ Initial Setup Complete (Driver fix required next).")





In [ ]:
# ---------------------------------------------------------------------------
# CELL 3: GPU DRIVER REPAIR (The "Hammer")
# ---------------------------------------------------------------------------
import os

VENV_PIP = "/kaggle/working/pyapi/venv/bin/pip"

print("🔧 Uninstalling incompatible PyTorch...")
!{VENV_PIP} uninstall -y torch torchvision torchaudio

print("🔧 Installing PyTorch (CUDA 11.8)...")
# We use the official PyTorch index for CUDA 11.8
!{VENV_PIP} install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --no-cache-dir

print("✅ PyTorch repaired. Ready for Inference.")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 3.5: DEPENDENCY BYPASS (The Final Patch)
# ---------------------------------------------------------------------------
import os

VENV_PIP = "/kaggle/working/pyapi/venv/bin/pip"

print("🔧 Forcing so-vits-svc-fork to accept our PyTorch version...")

# We reinstall the library without checking dependencies. 
# This makes it ignore the "Requires torch >= 2.8" rule.
!{VENV_PIP} install so-vits-svc-fork --no-deps --force-reinstall

print("✅ Bypass applied. You are ready to start the Server (Cell 4).")

In [ ]:
# ---------------------------------------------------------------------------
# ZROK SETUP
# ---------------------------------------------------------------------------
import os

# Download and install Zrok
if not os.path.exists("zrok"):
    print("⬇️ Downloading Zrok...")
    !wget https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !tar -xzf zrok_1.1.3_linux_amd64.tar.gz > /dev/null
    !chmod +x zrok

# Disable previous session if exists (cleanup)
!./zrok disable > /dev/null 2>&1

# Enable Zrok
print("🔗 Enabling Zrok...")
!./zrok enable --headless "$zrok_token"

In [ ]:
# Kill anything running on port 8009 to allow a clean restart
#!fuser -k 8009/tcp
#print("✅ Port 8009 cleared. You can now restart Cell 4.")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 4: RUN SERVER & TUNNEL (FINAL VENV EDITION)
# ---------------------------------------------------------------------------
import threading
import subprocess
import time
import os

SERVER_PORT = 8009
WORK_DIR = "/kaggle/working/pyapi" 
VENV_PYTHON = os.path.join(WORK_DIR, "venv", "bin", "python")

def run_pyapi_server():
    """Runs the FastAPI server using the VENV python with a clean environment"""
    
    # 1. Verify Python exists
    if not os.path.exists(VENV_PYTHON):
        print(f"❌ Critical Error: Python interpreter not found at {VENV_PYTHON}")
        return

    print(f"🚀 Starting PyAPI Server on port {SERVER_PORT} using venv...")
    
    # 2. Construct Command
    # We call the VENV python directly. 
    cmd = [
        VENV_PYTHON, "-m", "uvicorn", "main:app", 
        "--host", "0.0.0.0", 
        "--port", str(SERVER_PORT)
    ]
    
    # 3. Construct Isolated Environment
    # This is crucial: we strip Kaggle's global paths to prevent "sitecustomize" errors
    env = os.environ.copy()
    
    # Point to our venv
    env["VIRTUAL_ENV"] = os.path.join(WORK_DIR, "venv")
    # Prepend venv bin to PATH
    env["PATH"] = os.path.join(WORK_DIR, "venv", "bin") + ":" + env["PATH"]
    
    # CRITICAL: Remove PYTHONPATH so Kaggle's global libs don't interfere
    if "PYTHONPATH" in env:
        del env["PYTHONPATH"]

    # 4. Start Process
    process = subprocess.Popen(
        cmd, 
        cwd=WORK_DIR,  # Run inside /kaggle/working/pyapi
        env=env,       # Pass the clean environment
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    
    # 5. Log Output
    print("[PyAPI] Listening for logs...")
    try:
        for line in iter(process.stdout.readline, ''):
            clean_line = line.strip()
            if clean_line:
                # Filter for important messages to keep notebook clean
                if any(x in clean_line.lower() for x in ["error", "started", "exception", "failed", "uvicorn running"]):
                    print(f"[PyAPI] {clean_line}")
    except Exception as e:
        print(f"[PyAPI] Logger Error: {e}")

def start_zrok_tunnel():
    """Creates the public tunnel"""
    time.sleep(10) # Give venv/python ample time to warm up
    print("🚇 Opening Zrok Tunnel...")
    
    if os.path.exists("./zrok"):
        subprocess.run(["chmod", "+x", "./zrok"])
    
    # Start zrok share
    process = subprocess.Popen(
        ["./zrok", "share", "public", f"localhost:{SERVER_PORT}", "--headless"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Wait for tunnel registration
    time.sleep(10)
    
    # Get Public URL
    status_cmd = subprocess.run(["./zrok", "overview"], capture_output=True, text=True)
    print("\n" + "="*40)
    print(" 🌐 ZROK PUBLIC ACCESS ")
    print("="*40)
    print(status_cmd.stdout)
    print("="*40)

# Start Threads
thread_server = threading.Thread(target=run_pyapi_server, daemon=True)
thread_server.start()

thread_tunnel = threading.Thread(target=start_zrok_tunnel, daemon=True)
thread_tunnel.start()

In [ ]:
import time
print("✨ System operational. Running keep-alive loop.")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Shutting down.")